## ESPnet Text-to-Speech Demo


In [ ]:
from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from huggingface_hub import snapshot_download
from pathlib import Path

def download_vocoder(tag: str) -> str:
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)

    for file in repo_path.rglob("*.pkl"):
        print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError("No .pkl file found in the vocoder repo")
   

def init_tts(am_tag=None, vocoder_tag=None):
    voc_str = vocoder_tag
    if not voc_str:
        voc_str = "Griffin-Lim"
    am_str = am_tag

    print(f"\n===================================\nAM      = {am_str}")
    print(f"Vocoder = {voc_str}")
    print(f"===================================")

    vocoder_file=None
    if vocoder_tag:
        vocoder_file = download_vocoder(vocoder_tag)
        vocoder_tag = None
    
    tts = Text2Speech.from_pretrained(
        model_tag=am_tag,
        vocoder_tag=vocoder_tag,
        vocoder_file=vocoder_file,
        device="cpu"
    )
    return tts

print ("ESPnet initialized")    

### Select model

In [ ]:
am_tag_hf="Airenas/vdu-arn.fastspeech2.v01"

vocoder_tag_hf= "Airenas/vdu-arn.vocoder.style_melgan.v01"

tts_hf = init_tts(am_tag=am_tag_hf, vocoder_tag=vocoder_tag_hf)
tts_gl = init_tts(am_tag=am_tag_hf)

tts_list = [tts_hf, tts_gl]

print (f"\n\nREADY: {len(tts_list)} models loaded\n")  

### Enter text to read

In [ ]:
### Kai kurie tekstai paimti iš lrt.lt
texts = ["Sveiki, aš naujas lietuviškas balsas.", 
         "Sveiki! Aš naujas lietuviškas balsas!",
         "Sveiki? Ar aš naujas lietuviškas balsas?",
         "Aš esu labai gerai įrašytas.", 
         "Kaip Jums patinku?", 
         "Pernai maitinimo sektorių sukrėtė dešimtmečius veikusių restoranų ir kavinių bankrotai.", 
         "Šiomis dienomis orai Lietuvoje ims šilti, kris šlapdriba, sniegas, reikės pasisaugoti lijundros.", 
         "Tai savo „Facebook“ paskyroje sekmadienį vakare pranešė Ukrainos pirmasis vicepremjeras ir energetikos ministras Denysas Šmyhalis, skelbia „Ukrinform“."
        ]
for i, text in enumerate(texts):
    print(f"\n==========================================================================\nText = {text}")
    for it, tts in enumerate(tts_list):
        wav = tts(text)["wav"]
        filename = f"output_{i}_{it}.wav"
        sf.write(filename, wav.numpy(), tts.fs)
        audio_widget = Audio(filename)._repr_html_()
        display(HTML(f"<div style='display:flex; align-items:center; gap:10px;'><span>model {it}:</span>{audio_widget}</div>"))
